# Bundle Authoring — Companion Notebook

Mirrors the [Bundle Authoring Guide](https://gongjr0.github.io/SymbolicDSGE/latest/guides/bundle_authoring_guide/). Each section follows the same headings as the guide.

Run from `docs/assets/` so the relative path to `MODELS/POST82.yaml` resolves.

In [ ]:
import io

import numpy as np
import pandas as pd
from numpy import array, float64

from SymbolicDSGE import BundleBuilder, DSGESolver, Estimator, ModelParser
from SymbolicDSGE.bundle import ShockGeneration, SimSpec
from SymbolicDSGE.estimation.spec import (
    EstimationSpec,
    OptimizationResultMeta,
    PriorSpec,
)
from SymbolicDSGE.monte_carlo.spec import EdgeSpec, NodeSpec, PipelineSpec

## Solve a reference model

In [ ]:
parser = ModelParser("../../MODELS/POST82.yaml")
model, kalman = parser.get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(linearize=False)
sol = solver.solve(
    compiled,
    steady_state=array([0.0, 0.0, 0.0, 0.0, 0.0], dtype=float64),
)

## Specify the estimation tab

`EstimationSpec.from_targets` builds the spec from just the parameter names — `estimate=True` is flagged for each automatically. The synthetic `OptimizationResultMeta` below stands in for a real result so the demo bundle still ships a populated estimation tab; the fast-path cells later replace this manual construction with a live `Estimator` run.

In [ ]:
estimation_spec = EstimationSpec.from_targets(
    ["psi_pi", "psi_x"],
    method="map",
    initial={"psi_pi": 1.5, "psi_x": 0.5},
    bounds={"psi_pi": (1.0, 3.0), "psi_x": (0.0, 1.0)},
    priors={
        "psi_pi": PriorSpec(
            distribution="normal", parameters={"loc": 1.5, "scale": 0.25}
        ),
        "psi_x": PriorSpec(
            distribution="normal", parameters={"loc": 0.5, "scale": 0.2}
        ),
    },
    observables=["Infl", "Rate"],
    method_kwargs={"options": {"maxiter": 50}},
)

rng = np.random.default_rng(0)
observed = rng.standard_normal((40, 2))
estimation_result_meta = OptimizationResultMeta(
    kind="map",
    theta={"psi_pi": 1.48, "psi_x": 0.55},
    success=True,
    message="Optimization converged.",
    fun=-87.3,
    loglik=-85.1,
    logprior=-2.2,
    logpost=-87.3,
    nfev=124,
    nit=14,
)

## Build a Monte Carlo pipeline

In [ ]:
mc_pipeline = PipelineSpec(
    nodes=[
        NodeSpec(
            id="sim",
            step_type="simulation",
            name="DGP Simulation",
            params={"T": 100},
        ),
        NodeSpec(
            id="jb",
            step_type="jarque_bera",
            name="Normality Check",
            params={"source": "observables"},
        ),
    ],
    edges=[EdgeSpec(source="sim", target="jb")],
)

## Specify a simulation prefill

In [ ]:
simulation = SimSpec(
    role="reference",
    T=25,
    observables=True,
    shock_scale=1.0,
    shock_generation=ShockGeneration(
        dist="norm",
        seed=42,
        loc=0.0,
    ),
)

## Assemble and write

In [ ]:
bundle_path = (
    BundleBuilder(created_by="experiment-1")
    .add_model(
        "reference",
        model.source_yaml,
        compile_kwargs={"linearize": False},
    )
    .add_estimation(
        estimation_spec,
        result=estimation_result_meta,
        observed=observed,
        observable_names=["Infl", "Rate"],
    )
    .add_mc(mc_pipeline)
    .set_simulation(simulation)
    .write("experiment-1.sdsge")
)

print("Bundle written to:", bundle_path)

### Fast path: build spec and result from a real run

`PriorSpec` is the declarative form the bundle stores; `Prior` is the runtime form the `Estimator` evaluates. Materialize one from the other via `Estimator.make_prior`, then construct an `Estimator` directly so `to_spec` is reachable.

In [ ]:
prior_specs = {
    "psi_pi": PriorSpec(distribution="normal", parameters={"loc": 1.5, "scale": 0.25}),
    "psi_x": PriorSpec(distribution="normal", parameters={"loc": 0.5, "scale": 0.2}),
}
built_priors = {
    name: Estimator.make_prior(
        distribution=spec.distribution,
        parameters=dict(spec.parameters),
        transform=spec.transform,
    )
    for name, spec in prior_specs.items()
}

In [ ]:
estimator = Estimator(
    solver=solver,
    compiled=compiled,
    y=observed,
    estimated_params=["psi_pi", "psi_x"],
    priors=built_priors,
)
live_result = (
    estimator.map()
)  # OptimizationResult; switch to estimator.mcmc(...) for MCMC

spec_from_live = estimator.to_spec(method="map", priors=prior_specs)
print("Live theta:", live_result.theta)
print("Spec parameters:", [p.name for p in spec_from_live.parameters])

In [ ]:
real_run_path = (
    BundleBuilder()
    .add_model("reference", model.source_yaml)
    .add_estimation(
        spec_from_live,
        result=live_result,  # auto-projected via live_result.to_meta()
        observed=observed,
        observable_names=["Infl", "Rate"],
    )
    .write("real-run.sdsge")
)
print("Live-result bundle:", real_run_path)

## Add raw data alongside the model

In [ ]:
aux = pd.DataFrame(
    {
        "date": pd.date_range("2000-01-01", periods=40, freq="QS"),
        "gdp_growth": rng.standard_normal(40),
    }
)
csv_buffer = io.StringIO()
aux.to_csv(csv_buffer, index=False)

BundleBuilder().add_model("reference", model.source_yaml).add_raw_data(
    "auxiliary_series", csv_buffer.getvalue()
).write("with-raw-data.sdsge")

## Inspect the result

In [ ]:
!unzip -l experiment-1.sdsge